In [ ]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()

print(data.feature_names)
print(data.data.shape)          # (20640 houses, 8 features)
print(data.target[:5])          # price in units of $100,000 → 4.526 = $452,600

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

device = "cuda" if torch.cuda.is_available() else "cpu"

X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

# feature scales differ a lot (house age ~30, population ~1000+), so standardize them first
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)   # use the scale from train, don't refit

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)  # (N,) → (N, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)

model = nn.Sequential(
    nn.Linear(8, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1),            # 1 output = price
).to(device)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 20

# training
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch+1} | loss {total_loss / len(train_loader):.4f}")

# evaluation
model.eval()
with torch.no_grad():
    pred = model(X_test.to(device)).cpu()

mae = (pred - y_test).abs().mean().item()
print(f"\naverage prediction error: ${mae * 100_000:,.0f}")

for i in range(5):
    print(f"predicted ${pred[i].item() * 100_000:>9,.0f} | actual ${y_test[i].item() * 100_000:>9,.0f}")